# Grammar Scoring Engine for Spoken English

## Executive Summary

This project develops an automated **Grammar Scoring Engine** that predicts a continuous grammar proficiency score from a 45–60 second spoken-English audio recording.

The task is formulated as a regression problem. Each recording receives a score from **0 to 5**, representing the speaker's grammatical accuracy, sentence structure, syntactic control, and ability to use increasingly complex language.

### Final Performance

| Metric | Result |
|---|---:|
| Speaker-grouped CV RMSE | **0.5391** |
| Pearson correlation | **0.847** |
| All-training CV RMSE | **0.5260** |
| Training RMSE | **0.0961** |
| All-training RMSE | **0.0938** |
| Public leaderboard RMSE | **0.3728** |

The mean-prediction baseline has an RMSE of **1.0137**, showing a substantial improvement from the proposed system.

### Problem Statement

The objective is to predict the grammatical proficiency of a speaker from a 45–60 second spoken-English recording.

The target score follows the competition's grammar proficiency rubric:

| Score | Interpretation |
|---:|---|
| 1 | Very limited control of grammar and sentence structure |
| 2 | Basic structures with frequent grammatical errors |
| 3 | Reasonable grammatical control with noticeable errors |
| 4 | Strong grammatical control with only minor errors |
| 5 | Highly accurate and effective use of complex grammar |

Because the target reflects spoken proficiency rather than isolated grammatical errors, the system combines **linguistic, semantic, acoustic, and temporal speech representations**.

### Overall Pipeline

**Audio → Whisper-medium ASR → Transcript + speech representations → Feature engineering → Individual regressors → OOF ensemble → Temporal attention refinement → Final grammar score**

The final feature space combines:

1. **Fluency and lexical features** — speech rate, pauses, fillers, repetitions, lexical diversity, sentence statistics, and ASR-related features.
2. **Grammar and language-model features** — language-model scores, CoLA acceptability features, and LanguageTool error statistics.
3. **Signal-quality features** — acoustic measurements used to identify potentially unusable recordings.
4. **Speech representations** — wav2vec2 and Whisper encoder representations.
5. **Semantic representation** — sentence-level embeddings.
6. **Temporal speech representation** — frame-level Whisper-medium features processed through an attention-pooling prediction head.

The tabular and embedding features are modeled using **Ridge Regression, LightGBM, and Support Vector Regression (SVR)**. Their out-of-fold predictions are combined using non-negative blending. A temporal attention head is then used as a complementary prediction component.

### Final Model

The final configuration uses:

- **Whisper-medium** for transcription
- Speaker-grouped cross-validation
- Ridge Regression
- LightGBM
- Support Vector Regression
- Sentence and speech embeddings
- Temporal attention pooling
- Non-negative prediction blending
- Output clipping to the valid **0–5** range

The final speaker-grouped CV RMSE is **0.5391**, with a Pearson correlation of **0.847**.

### Key Findings

- Speaker-grouped validation is more conservative than random validation, highlighting the importance of controlling for speaker-specific information.
- The multimodal feature representation performs better when combined through an ensemble than when individual regressors are used independently.
- The base ensemble improves from the best individual model through complementary predictions.
- The temporal attention head further improves the grouped-CV RMSE from **0.5540 to 0.5391**, a gain of **0.0149**.
- The public leaderboard score improved from **0.3784 to 0.3728** after the final refinement.

### Important Evaluation Note

The **training RMSE of 0.0961** is reported because it is explicitly required by the competition. However, it is not treated as the primary estimate of generalization. The speaker-grouped CV RMSE of **0.5391** is the more conservative performance estimate because it evaluates the model on held-out speaker groups.

### Limitations

- ASR errors can hide or alter grammatical mistakes.
- Whisper performance can vary across accents and recording conditions.
- Speaker groups are inferred from acoustic representations rather than provided explicitly.
- The substantially lower training RMSE than grouped-CV RMSE indicates overfitting risk.
- Feature and model selection were performed using the same grouped validation framework, so the reported CV estimate may be somewhat optimistic.

In [ ]:
!pip -q install faster-whisper language_tool_python lightgbm sentence-transformers

In [ ]:
import os, re, json, time, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import librosa, torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from scipy.optimize import nnls
from sklearn.model_selection import StratifiedGroupKFold, KFold, cross_val_predict
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
warnings.filterwarnings('ignore'); sns.set_theme()
SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
rmse=lambda a,b: float(np.sqrt(mean_squared_error(a,b)))
print('device:',DEVICE)

## 1. Config, data loading and integrity checks

The dataset contains:

- **769 labelled training recordings**
- **216 test recordings**
- WAV audio files
- `train.csv`
- `test.csv`
- `sample_submission.csv`

Each recording is approximately 45–60 seconds long.

### Data Integrity

The same filenames can occur in both the training and test directories.
Therefore, filenames are not treated as globally unique identifiers.

Instead, each recording is identified by:

**split + filename**

The notebook explicitly indexes the training and test directories separately and
checks whether files with matching names are actually different recordings by
comparing their file sizes.

This prevents accidental mixing of train/test recordings during preprocessing.

In [ ]:
DATA_DIR  = '/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final'
TRAIN_CSV, TEST_CSV, SAMPLE_SUB = f'{DATA_DIR}/train.csv', f'{DATA_DIR}/test.csv', f'{DATA_DIR}/sample_submission.csv'
CACHE='/kaggle/temp/cache'; os.makedirs(CACHE,exist_ok=True)     # outside /kaggle/working so the output folder stays clean
WHISPER_SIZE='medium'
PCA_DIM=32; N_SPLITS=5

train=pd.read_csv(TRAIN_CSV); test=pd.read_csv(TEST_CSV); sub_ex=pd.read_csv(SAMPLE_SUB)
FILE_COL=train.columns[0]; LABEL_COL=train.columns[1]

# Index audio PER SPLIT: train/ and test/ reuse the same filenames for different recordings.
def build_index(split):
    d=f'{DATA_DIR}/{split}'
    return {f:os.path.join(r,f) for r,_,fs in os.walk(d) for f in fs if f.endswith('.wav')}
IDX={s:build_index(s) for s in ['train','test']}
fix=lambda n: n if n.endswith('.wav') else n+'.wav'

ALL=pd.concat([pd.DataFrame({'split':'train','name':train[FILE_COL].map(fix),'y':train[LABEL_COL].astype(float)}),
               pd.DataFrame({'split':'test','name':test[FILE_COL].map(fix),'y':np.nan})],ignore_index=True)
ALL['path']=[IDX[s].get(n) for s,n in zip(ALL.split,ALL.name)]
ALL['key']=ALL.split+'__'+ALL.name
IS_TR=(ALL.split=='train').values; IS_TE=~IS_TR
print('rows:',len(ALL),'| missing audio:',ALL.path.isna().sum())

dups=sorted(set(IDX['train'])&set(IDX['test']))
diff=sum(os.path.getsize(IDX['train'][n])!=os.path.getsize(IDX['test'][n]) for n in dups)
print(f'filenames present in both splits: {len(dups)} | of these, different file size (=different recording): {diff}')

### EDA Observations

The target distribution is inspected before modeling to identify class imbalance
and the presence of exact-zero labels.

Audio duration is also checked to verify that the recordings follow the expected
45–60 second range.

The exact-zero samples are investigated separately rather than automatically
assuming that a zero score represents extremely poor grammar.

In [ ]:
y_tr=ALL.loc[IS_TR,'y']
fig,ax=plt.subplots(1,2,figsize=(11,3.5))
sns.countplot(x=y_tr.round().astype(int),ax=ax[0]); ax[0].set_title('Label distribution (rounded)')
durs=[librosa.get_duration(path=p) for p in tqdm(ALL.path.sample(150,random_state=0))]
ax[1].hist(durs,bins=20); ax[1].set_title('Clip duration (s), 150 random clips'); plt.show()
print(y_tr.describe()); print('exact zeros:',(y_tr==0).sum())

In [ ]:
print(f"Training samples: {len(y_tr)}")
print(f"Mean score: {y_tr.mean():.3f}")
print(f"Std score: {y_tr.std():.3f}")
print(f"Minimum score: {y_tr.min():.3f}")
print(f"Maximum score: {y_tr.max():.3f}")
print(f"Exact-zero labels: {(y_tr == 0).sum()}")

## 2. Signal-quality features (one pass over the audio)

In [ ]:
load=lambda p: librosa.load(p,sr=16000,mono=True)[0]
def quality_feats(y):
    S=np.abs(librosa.stft(y,n_fft=1024,hop_length=256))
    flat=librosa.feature.spectral_flatness(S=S)[0]; rms=librosa.feature.rms(S=S,frame_length=1024)[0]
    thr=0.1*np.percentile(rms,95)
    return dict(q_flat_mean=flat.mean(), q_flat_med=np.median(flat), q_rms_mean=rms.mean(), q_rms_std=rms.std(),
                q_speech_ratio=(rms>thr).mean(), q_peak=np.abs(y).max(),
                q_dyn_db=20*np.log10((np.percentile(rms,90)+1e-8)/(np.percentile(rms,10)+1e-8)),
                q_dur=len(y)/16000)
qp=f'{CACHE}/quality.pkl'
if os.path.exists(qp): Q=pd.read_pickle(qp)
else:
    Q=pd.DataFrame([quality_feats(load(p)) for p in tqdm(ALL.path)]); Q.to_pickle(qp)
Q.describe().T[['mean','std','min','max']]

## 3. Corrupted-audio gate
Do exact-zero labels correspond to unusable audio rather than ungrammatical speech? We test it instead of assuming it: if a one-split decision tree on signal-quality features separates zeros in cross-validation, zeros are gated.

### Zero-score Investigation

Some training samples have an exact target value of zero.

Rather than assuming that these recordings represent extremely poor grammar,
we test whether they are distinguishable using basic signal-quality features.

A shallow decision tree is evaluated using cross-validation. The gate is enabled
only when the zero/non-zero distinction is sufficiently reliable.

This prevents an unsupported assumption about the meaning of the zero label from
being built directly into the model.

In [ ]:
zero=(ALL.y==0).values
GATE_ON=False; gate_clf=None
if zero.sum()>=5:
    Xq=Q[IS_TR].values; zq=zero[IS_TR].astype(int)
    gate_clf=DecisionTreeClassifier(max_depth=1,random_state=SEED)
    cvp=cross_val_predict(gate_clf,Xq,zq,cv=5)
    acc=(cvp==zq).mean(); errs=(cvp!=zq).sum()
    gate_clf.fit(Xq,zq); feat=Q.columns[gate_clf.tree_.feature[0]]; thr=gate_clf.tree_.threshold[0]
    print(f'gate feature: {feat} @ {thr:.4f} | 5-fold CV accuracy {acc:.4f} | misclassified {errs}/{len(zq)}')
    GATE_ON = errs<=1
    fig,ax=plt.subplots(figsize=(6,3.5))
    ax.hist(Q.loc[IS_TR&~zero,feat],bins=40,alpha=.6,label='label>0'); ax.hist(Q.loc[IS_TR&zero,feat],bins=40,alpha=.6,label='label=0')
    ax.axvline(thr,color='r',ls='--'); ax.legend(); ax.set_title(f'{feat}'); plt.show()
print('GATE_ON =',GATE_ON)
GATED=np.zeros(len(ALL),bool)
if GATE_ON: GATED=gate_clf.predict(Q.values).astype(bool)
print('gated in train:',GATED[IS_TR].sum(),'| gated in test:',GATED[IS_TE].sum())
# rows used to train grammar models: labelled training rows that are not exact zeros
G_IDX=np.where(IS_TR&~zero)[0] if GATE_ON else np.where(IS_TR)[0]
print('grammar-model training rows:',len(G_IDX))

## 4. Transcription (Whisper-medium, word timestamps and confidences)
### Automatic Speech Recognition

Whisper-medium is used to convert the speech signal into text.

Word timestamps and word-level confidence values are retained because the
transcript is not used only as text: timing and confidence are also important
inputs to the fluency feature block.

A prompt containing common fillers and repetitions is supplied to reduce the
tendency of transcription to remove speech disfluencies such as "um", "uh",
and repeated words.

### Model Selection

Whisper-medium was selected after comparing it against the smaller Whisper
configuration.

The medium model produced a grouped-CV RMSE of **0.5532**, compared with
**0.5556** for the smaller configuration, while also improving the public
score.

The improvement in grouped CV was modest, so the decision was supported by
both validation and leaderboard behavior.

In [ ]:
from faster_whisper import WhisperModel
asr=WhisperModel(WHISPER_SIZE,device=DEVICE,compute_type='float16' if DEVICE=='cuda' else 'int8')
PROMPT='Umm, let me think, like, hmm... Okay, so, uh, I was, I was going to say that, you know.'   # keeps fillers/repeats
def transcribe(key,path):
    cp=f'{CACHE}/asr_{WHISPER_SIZE}_{key}.json'
    if os.path.exists(cp): return json.load(open(cp))
    y=load(path)
    segs,_=asr.transcribe(y,language='en',word_timestamps=True,initial_prompt=PROMPT,beam_size=5,condition_on_previous_text=False)
    out=dict(words=[dict(w=w.word.strip(),s=w.start,e=w.end,p=w.probability) for sg in segs for w in sg.words],duration=len(y)/16000)
    json.dump(out,open(cp,'w')); return out
ASR=[transcribe(k,p) for k,p in tqdm(zip(ALL.key,ALL.path),total=len(ALL))]
TEXT=[' '.join(w['w'] for w in a['words']) for a in ASR]
del asr; torch.cuda.empty_cache()           # free the GPU for the models below
print(TEXT[0][:300])

## 5. Feature Engineering

The model uses several complementary feature families.


### 5.1 Fluency and Lexical Features

These features are extracted from the Whisper transcript and word-level
timestamps.

Examples include:

- Words per minute
- Pause duration
- Number of long pauses
- Filler-word rate
- Repetition rate
- Type-token ratio
- Bigram diversity
- Sentence length
- Word length
- ASR confidence statistics

These features capture fluency, lexical richness and structural complexity.

In [ ]:
# --- Block 'basic': fluency / lexical / structure features from transcript + word timings
FILL={'um','uh','umm','uhh','er','erm','hmm','mm','ah'}
def basic_feats(a):
    W=a['words']; toks=[re.sub(r"[^a-z']",'',w['w'].lower()) for w in W]; toks=[t for t in toks if t]; n=max(len(toks),1); dur=max(a['duration'],1e-6)
    gaps=np.array([W[i+1]['s']-W[i]['e'] for i in range(len(W)-1)]) if len(W)>1 else np.array([0.])
    pr=np.array([w['p'] for w in W]) if W else np.array([0.])
    sents=[s for s in re.split(r'[.!?]+',' '.join(w['w'] for w in W)) if s.strip()]; sl=np.array([len(s.split()) for s in sents]) if sents else np.array([0])
    reps=sum(toks[i]==toks[i-1] for i in range(1,len(toks))); bi=list(zip(toks,toks[1:]))
    return dict(b_nwords=len(toks), b_wpm=len(toks)/dur*60, b_ttr=len(set(toks))/n, b_fill=sum(t in FILL for t in toks)/n,
        b_rep=reps/n, b_bigram_div=len(set(bi))/max(len(bi),1), b_pause_mean=gaps.mean(), b_pause_max=gaps.max(),
        b_pause_long=(gaps>0.5).sum()/dur*60, b_art_rate=len(toks)/max(dur-gaps[gaps>0.25].sum(),1e-6),
        b_p_mean=pr.mean(), b_p_min=pr.min(), b_p_low=(pr<0.5).mean(), b_p_std=pr.std(),
        b_nsent=len(sents), b_sl_mean=sl.mean(), b_sl_std=sl.std(), b_sl_max=sl.max(),
        b_wlen=np.mean([len(t) for t in toks]) if toks else 0, b_long=np.mean([len(t)>=7 for t in toks]) if toks else 0)
B=pd.DataFrame([basic_feats(a) for a in ASR]); B.shape

### 5.2 Language-Model Features

Three complementary grammar-related signals are extracted:

- GPT-2 negative log-likelihood
- RoBERTa-CoLA acceptability scores
- LanguageTool grammar/error statistics

These provide model-based estimates of how linguistically well-formed the
transcribed speech is.

In [ ]:
# --- Block 'lm': language-model grammar signals
# (1) GPT-2 negative log-likelihood per text chunk
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, pipeline
tok=GPT2TokenizerFast.from_pretrained('gpt2'); gpt=GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE).eval()
@torch.no_grad()
def gpt_feats(text):
    ids=tok(text,return_tensors='pt').input_ids[0]; nll=[]
    for i in range(0,len(ids),128):
        c=ids[i:i+128].unsqueeze(0).to(DEVICE)
        if c.shape[1]>=8: nll.append(gpt(c,labels=c).loss.item())
    nll=np.array(nll) if nll else np.array([0.])
    return dict(l_gpt_mean=nll.mean(), l_gpt_max=nll.max(), l_gpt_std=nll.std())
# (2) CoLA acceptability of each sentence (RoBERTa fine-tuned on CoLA); LABEL_1 = acceptable
cola=pipeline('text-classification',model='textattack/roberta-base-CoLA',device=0 if DEVICE=='cuda' else -1,top_k=None,truncation=True,max_length=128)
def sentences(text):
    s=[x.strip() for x in re.split(r'(?<=[.!?])\s+',text) if len(x.split())>=3]
    if len(s)<3:   # unpunctuated ASR: fall back to 15-word windows
        w=text.split(); s=[' '.join(w[i:i+15]) for i in range(0,len(w),15)]
    return s
def cola_feats(text):
    s=sentences(text)
    if not s: return dict(l_cola_mean=0,l_cola_min=0,l_cola_bad=1,l_cola_std=0)
    p=np.array([next(d['score'] for d in r if d['label']=='LABEL_1') for r in cola(s)])
    return dict(l_cola_mean=p.mean(), l_cola_min=p.min(), l_cola_bad=(p<0.5).mean(), l_cola_std=p.std())
# (3) LanguageTool rule matches (needs Java; skipped gracefully if unavailable)
try:
    import language_tool_python; LT=language_tool_python.LanguageTool('en-US')
except Exception as e:
    LT=None; print('LanguageTool unavailable:',repr(e)[:120])
def lt_feats(text):
    if LT is None: return dict(l_lt_rate=0,l_lt_gram=0,l_lt_total=0)
    m=LT.check(text); n=max(len(text.split()),1)
    def rid(x): return str(getattr(x,'rule_id',None) or getattr(x,'ruleId','') or '')
    def cat(x): return str(getattr(x,'category','') or '')
    gram=sum((cat(x) in ('GRAMMAR','TYPOS')) or ('AGREEMENT' in rid(x)) for x in m)
    return dict(l_lt_rate=len(m)/n*100, l_lt_gram=gram/n*100, l_lt_total=len(m))
print(lt_feats(TEXT[0]))          # quick sanity check

lp=f'{CACHE}/lm_{WHISPER_SIZE}.pkl'
if os.path.exists(lp): L=pd.read_pickle(lp)
else:
    L=pd.DataFrame([{**gpt_feats(t),**cola_feats(t),**lt_feats(t)} for t in tqdm(TEXT)]); L.to_pickle(lp)
L.describe().T[['mean','std']]

### 5.3 Acoustic and Speech Representations

The system also extracts pretrained speech representations from:

- wav2vec2-base
- Whisper-medium encoder layers

Mean and variance statistics are used to obtain fixed-size representations.

### 5.4 Semantic Text Representation

The transcript is represented using an MPNet sentence embedding.

This provides a dense semantic representation that complements the handcrafted
linguistic features.

In [ ]:
# --- Embedding blocks: audio (wav2vec2-base, Whisper-medium encoder) and text (sentence embedding)
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model, WhisperFeatureExtractor, WhisperModel
w2v_fe=Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base'); w2v=Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base').to(DEVICE).eval()
WH_DT=torch.float32
wh_fe=WhisperFeatureExtractor.from_pretrained('openai/whisper-medium')
wh_enc=WhisperModel.from_pretrained('openai/whisper-medium',torch_dtype=WH_DT).encoder.to(DEVICE).eval()
ms=lambda h: torch.cat([h.mean(0),h.std(0)]).float().cpu().numpy()
@torch.no_grad()
def audio_embs(y):
    x=w2v_fe(y,sampling_rate=16000,return_tensors='pt').input_values.to(DEVICE)
    hs=w2v(x,output_hidden_states=True).hidden_states
    out=dict(w2v_early=ms(hs[3][0]),w2v_mid=ms(hs[6][0]),w2v_last=ms(hs[12][0]))   # layer 3 mainly encodes speaker/channel -> used only for speaker grouping
    mids,lasts,wts=[],[],[]
    for i in range(0,len(y),480000):                     # Whisper encoder takes 30 s windows
        ch=y[i:i+480000]
        if len(ch)<16000: continue
        feats=wh_fe(ch,sampling_rate=16000,return_tensors='pt').input_features.to(DEVICE,dtype=WH_DT)
        o=wh_enc(feats,output_hidden_states=True)
        n=min(1500,int(np.ceil(len(ch)/320))); hs2=o.hidden_states
        mids.append(hs2[len(hs2)//2][0,:n].float().mean(0).cpu().numpy()); lasts.append(o.last_hidden_state[0,:n].float().mean(0).cpu().numpy()); wts.append(n)
    out['wh_mid']=np.average(mids,axis=0,weights=wts); out['wh_last']=np.average(lasts,axis=0,weights=wts)
    return out
ep=f'{CACHE}/audio_embs.npz'
if os.path.exists(ep): Z=dict(np.load(ep))
else:
    R=[audio_embs(load(p)) for p in tqdm(ALL.path)]
    Z={k:np.stack([r[k] for r in R]) for k in R[0]}; np.savez(ep,**Z)
from sentence_transformers import SentenceTransformer
Z['sent']=SentenceTransformer('all-mpnet-base-v2',device=DEVICE).encode(TEXT,batch_size=32,show_progress_bar=True)
print({k:v.shape for k,v in Z.items()})

### 5.5 Semantic Representation

Sentence-level embeddings provide a compact representation of the transcript's semantic and linguistic content.

### 5.6 Temporal Representation

The final temporal component uses frame-level Whisper-medium representations with a sequence length of 150 and embedding dimension 1024.

An attention-pooling prediction head learns to emphasize informative portions of the spoken sequence instead of treating all temporal regions equally.

The resulting combined feature matrix contains 11,558 features for 985 recordings.

In [ ]:
# --- Extra encoder layers (Whisper-small layers 3, 9; Whisper-medium layers 6, 12, 18, 24), mean-pooled over time
@torch.no_grad()
def whisper_layers(model_name, layers):
    fe=WhisperFeatureExtractor.from_pretrained(model_name)
    dt=torch.float16 if DEVICE=='cuda' else torch.float32
    enc=WhisperModel.from_pretrained(model_name,torch_dtype=dt).encoder.to(DEVICE).eval()
    out={l:[] for l in layers}
    for p in tqdm(ALL.path,desc=model_name):
        y=load(p); acc={l:[] for l in layers}; wts=[]
        for i in range(0,len(y),480000):                      # 30 s windows
            ch=y[i:i+480000]
            if len(ch)<16000: continue
            x=fe(ch,sampling_rate=16000,return_tensors='pt').input_features.to(DEVICE,dtype=dt)
            hs=enc(x,output_hidden_states=True).hidden_states
            n=min(1500,int(np.ceil(len(ch)/320)))
            for l in layers: acc[l].append(hs[l][0,:n].float().mean(0).cpu().numpy())
            wts.append(n)
        for l in layers: out[l].append(np.average(acc[l],axis=0,weights=wts))
    del enc; torch.cuda.empty_cache()
    return {l:np.stack(v) for l,v in out.items()}
wp=f'{CACHE}/whisper_layers.npz'
if os.path.exists(wp): NEWZ=dict(np.load(wp))
else:
    s=whisper_layers('openai/whisper-small',[3,9])           # 25% and 75% depth (12 layers)
    m=whisper_layers('openai/whisper-medium',[6,12,18,24])   # 24 layers
    NEWZ={**{f'whs_l{l}':v for l,v in s.items()},**{f'whm_l{l}':v for l,v in m.items()}}
    np.savez(wp,**NEWZ)
print({k:v.shape for k,v in NEWZ.items()})

In [ ]:
# Assemble one DataFrame; COLS maps block name -> its columns
parts={'basic':B,'lm':L,'quality':Q}
for k,v in {**{k:Z[k] for k in ['w2v_mid','w2v_last','wh_mid','wh_last','sent']},**NEWZ}.items():
    parts[k]=pd.DataFrame(v,columns=[f'{k}_{i}' for i in range(v.shape[1])])
COLS={k:list(v.columns) for k,v in parts.items()}
X_ALL=pd.concat(parts.values(),axis=1).replace([np.inf,-np.inf],np.nan).fillna(0)

# ---- Final block configuration ----
TAB=['basic','lm','quality']
OLD_EMB=['w2v_mid','w2v_last','wh_mid','wh_last','sent']
S_NEW=['whs_l3','whs_l9']; M_NEW=['whm_l6','whm_l12','whm_l18','whm_l24']
EMB =OLD_EMB+S_NEW+M_NEW     # embedding blocks (scaled + PCA); used by ridge_emb
ALLB=TAB+OLD_EMB             # blocks used by ridge_all / lgbm_all / svr_all
print(X_ALL.shape)

## 6. Speaker-grouped validation
If one candidate recorded several clips, random K-fold lets the model see the same voice in train and validation. Speaker identity is inferred from low-layer wav2vec2 statistics (channel/speaker information) with agglomerative clustering (about 3 clips per group, an assumption), then used for `StratifiedGroupKFold`. The within-group label spread is reported, and random vs grouped folds are compared on the same model.

### Strategy

Random cross-validation can overestimate performance when multiple recordings
are associated with the same speaker.

If recordings from the same speaker appear in both the training and validation
folds, the model may exploit speaker-specific acoustic characteristics instead
of learning general grammatical proficiency.

To reduce this risk, speaker groups are inferred from low-level wav2vec2
representations using PCA followed by agglomerative clustering.

These inferred groups are then used with StratifiedGroupKFold.

### Random vs Speaker-grouped Validation

For the Ridge model using all base feature blocks:

- Random-fold RMSE: **0.5436**
- Speaker-grouped RMSE: **0.5783**

The degradation under grouped validation demonstrates why random splitting would
give an overly optimistic estimate for this dataset.

Therefore, all final model comparisons use speaker-grouped cross-validation.

In [ ]:
E=StandardScaler().fit_transform(Z['w2v_early'][G_IDX]); E=PCA(20,random_state=SEED).fit_transform(E)
n_groups=max(len(G_IDX)//3,2)
grp=AgglomerativeClustering(n_clusters=n_groups,linkage='ward').fit_predict(E)
yg=ALL.y.values[G_IDX]
within=np.mean([yg[grp==g].std() for g in np.unique(grp) if (grp==g).sum()>1])
print(f'groups: {n_groups} | mean size {len(grp)/n_groups:.2f} | mean within-group label std {within:.3f} vs overall {yg.std():.3f}')
folds=list(StratifiedGroupKFold(N_SPLITS,shuffle=True,random_state=SEED).split(G_IDX,np.round(yg).astype(int),grp))
rand_folds=list(KFold(N_SPLITS,shuffle=True,random_state=SEED).split(G_IDX))
Xg=X_ALL.iloc[G_IDX].reset_index(drop=True); Xte=X_ALL[IS_TE].reset_index(drop=True)
LO,HI=float(yg.min()),5.0; clip=lambda p: np.clip(p,LO,HI)

In [ ]:
def make_pipe(blocks,final):
    tr=[(b,make_pipeline(StandardScaler(),PCA(PCA_DIM,random_state=SEED)) if b in EMB else StandardScaler(),COLS[b]) for b in blocks]
    return make_pipeline(ColumnTransformer(tr),final)
ridge=lambda: RidgeCV(alphas=np.logspace(-1,3.5,20))
def cv_oof(make_est,fl,Xte_=None):
    oof=np.zeros(len(yg)); tp=np.zeros(len(Xte)) if Xte_ is not None else None
    for tr,va in fl:
        m=make_est().fit(Xg.iloc[tr],yg[tr]); oof[va]=m.predict(Xg.iloc[va])
        if Xte_ is not None: tp+=m.predict(Xte_)/len(fl)
    return oof,tp
# Leakage check: identical model, random vs grouped folds
o_r,_=cv_oof(lambda: make_pipe(ALLB,ridge()),rand_folds); o_g,_=cv_oof(lambda: make_pipe(ALLB,ridge()),folds)
RG=(rmse(yg,clip(o_r)),rmse(yg,clip(o_g)))
print(f'Ridge, all blocks | random-fold RMSE {RG[0]:.4f} | speaker-grouped RMSE {RG[1]:.4f}')

## 7. Ablation: what does each feature block add? (Ridge, speaker-grouped CV)

### Ablation Interpretation

The ablation experiments measure whether each feature family provides useful
information beyond the others.

The analysis is used to determine:

1. Which feature blocks provide complementary information.
2. Whether pretrained speech representations improve over transcript-only
   features.
3. Whether additional Whisper encoder layers provide useful information.
4. Whether any individual block can be removed without materially affecting
   performance.

Differences smaller than approximately **0.005 RMSE** are treated cautiously
because they may be within experimental variation.

In [ ]:
rows=[('mean predictor',rmse(yg,np.full_like(yg,yg.mean())))]
for b in TAB+OLD_EMB:
    o,_=cv_oof(lambda b=b: make_pipe([b],ridge()),folds); rows.append((f'only {b}',rmse(yg,clip(o))))
cum=[]
for step in [['basic'],['lm'],['quality'],['w2v_mid','w2v_last'],['wh_mid','wh_last'],['sent']]:
    cum+=step; o,_=cv_oof(lambda c=list(cum): make_pipe(c,ridge()),folds); rows.append(('cumulative: '+'+'.join(cum),rmse(yg,clip(o))))
ABL=pd.DataFrame(rows,columns=['feature set','CV RMSE']).round(4); ABL.to_csv('ablation.csv',index=False)
pd.set_option('display.max_colwidth',120); ABL

In [ ]:
# Extra encoder layers: do they help beyond the base blocks? Also PCA dimension.
def score(blocks):
    o,_=cv_oof(lambda: make_pipe(blocks,ridge()),folds); return rmse(yg,clip(o))
exps={'base blocks':TAB+OLD_EMB,
      '+ whisper-small extra layers':TAB+OLD_EMB+S_NEW,
      '+ whisper-medium layers':TAB+OLD_EMB+M_NEW,
      '+ both':TAB+OLD_EMB+S_NEW+M_NEW,
      'medium layers only (+TAB)':TAB+M_NEW}
R=pd.DataFrame([(k,round(score(v),4)) for k,v in exps.items()],columns=['feature set','CV RMSE']); display(R)
for d in [16,32,64]:
    PCA_DIM=d; print('PCA_DIM',d,'->',round(score(TAB+OLD_EMB+S_NEW+M_NEW),4))
PCA_DIM=32

In [39]:
del asr; torch.cuda.empty_cache() 
# Drop-one analysis on the full block set (differences below ~0.005 are treated as noise)
BEST=TAB+OLD_EMB+S_NEW+M_NEW
base=score(BEST); print('full set',round(base,4))
for b in BEST:
    print(f'drop {b:8s} -> {score([x for x in BEST if x!=b]):.4f}')

NameError: name 'asr' is not defined

## 8. Final models and blend

Four regression models are evaluated:

### Ridge Regression

Provides a strongly regularized linear baseline over the engineered feature
representation.

### LightGBM

Captures nonlinear relationships and interactions between the engineered
features.

### Support Vector Regression

Provides a nonlinear kernel-based regression model with a different inductive
bias from LightGBM.

### Embedding Ridge

A dedicated Ridge model is trained over the embedding representation.

The individual predictions are subsequently combined using non-negative least
squares on out-of-fold predictions.

In [ ]:
lgbm=lambda: lgb.LGBMRegressor(n_estimators=400,learning_rate=0.03,num_leaves=8,min_child_samples=15,subsample=0.8,subsample_freq=1,
                               colsample_bytree=0.6,reg_lambda=5,random_state=SEED,verbose=-1)
svr=lambda: make_pipeline(StandardScaler(),SVR(C=2.0,epsilon=0.15,gamma='scale'))
def ct_all(): return ColumnTransformer([(b,make_pipeline(StandardScaler(),PCA(PCA_DIM,random_state=SEED)) if b in EMB else StandardScaler(),COLS[b]) for b in ALLB])
MODELS={'ridge_all':lambda: make_pipe(ALLB,ridge()),
        'ridge_emb':lambda: make_pipe(EMB,ridge()),
        'lgbm_all' :lambda: make_pipeline(ct_all(),lgbm()),
        'svr_all'  :lambda: make_pipeline(ct_all(),svr())}
OOF,TP,FULL_TR,FULL_TE={}, {}, {}, {}
for k,mk in MODELS.items():
    OOF[k],TP[k]=cv_oof(mk,folds,Xte); m=mk().fit(Xg,yg); FULL_TR[k]=m.predict(Xg); FULL_TE[k]=m.predict(Xte)
    print(f'{k:10s} CV RMSE {rmse(yg,clip(OOF[k])):.4f}')

def blend_oof(names_):
    O_=np.column_stack([OOF[k] for k in names_]); w_,_=nnls(O_,yg); w_/=w_.sum(); return clip(O_@w_),w_
def blend_full(names_):
    o,w_=blend_oof(names_)
    t=clip(np.column_stack([FULL_TR[k] for k in names_])@w_)
    te=clip(0.5*(np.column_stack([TP[k] for k in names_])@w_)+0.5*(np.column_stack([FULL_TE[k] for k in names_])@w_))
    return o,t,te,dict(zip(names_,w_.round(3)))
base_names=['ridge_all','ridge_emb','lgbm_all','svr_all']
BASE=blend_full(base_names); oof_bl,tr_bl,te_bl=BASE[:3]
print('blend weights:',BASE[3]); print(f'BASE BLEND CV RMSE = {rmse(yg,oof_bl):.4f}')

## 8b. Temporal model: attention-pooling head on frame-level Whisper features

### Temporal Attention Refinement

Mean pooling converts a sequence of Whisper encoder states into a single
fixed-size representation, but this can discard information about **when**
important speech patterns occur.

To address this limitation, a small attention-pooling network is trained over
150 temporal Whisper-medium frame representations.

The network:

1. Normalizes the encoder features.
2. Projects them into a lower-dimensional representation.
3. Applies temporal attention.
4. Combines attention-weighted and mean-pooled representations.
5. Produces a scalar grammar prediction.

The Whisper encoder itself remains frozen.

### Selection Criterion

The attention head is retained only when it improves the speaker-grouped
ensemble CV RMSE by at least **0.005**.

### Result

| Configuration | Grouped CV RMSE |
|---|---:|
| Best individual model | 0.5673 |
| Base ensemble | **0.5536** |
| Ensemble + attention head | **0.5383** |
| Attention head alone | 0.6503 |


In [ ]:
T_FR=150
@torch.no_grad()
def frame_feats(y,T=T_FR):
    seq=[]
    for i in range(0,len(y),480000):                         # 30 s windows
        ch=y[i:i+480000]
        if len(ch)<16000: continue
        x=wh_fe(ch,sampling_rate=16000,return_tensors='pt').input_features.to(DEVICE,dtype=WH_DT)
        seq.append(wh_enc(x).last_hidden_state[0,:min(1500,int(np.ceil(len(ch)/320)))].float())
    h=torch.cat(seq,0).T.unsqueeze(0)                         # (1, D, frames)
    return F.adaptive_avg_pool1d(h,T)[0].T.half().cpu().numpy()   # (T, D)
fp=f'{CACHE}/frames_medium_T{T_FR}.npy'
if os.path.exists(fp): FR=np.load(fp)
else:
    FR=np.stack([frame_feats(load(p)) for p in tqdm(ALL.path)]); np.save(fp,FR)
print(FR.shape,FR.dtype,f'{FR.nbytes/1e9:.2f} GB')
FR_t=torch.tensor(FR,device=DEVICE)                           # fp16 on the device
FR_g =FR_t[torch.as_tensor(G_IDX,device=DEVICE)]              # grammar-model training rows (same order as yg)
FR_te=FR_t[torch.as_tensor(np.where(IS_TE)[0],device=DEVICE)] # test rows (same order as Xte)

In [ ]:
class Head(nn.Module):
    def __init__(s,d_in=1024,d=128,p=0.3,layers=0):
        super().__init__()
        s.norm=nn.LayerNorm(d_in); s.proj=nn.Linear(d_in,d); s.drop=nn.Dropout(p)
        s.pos=nn.Parameter(torch.zeros(1,T_FR,d))
        s.enc=(nn.TransformerEncoder(nn.TransformerEncoderLayer(d,4,2*d,p,batch_first=True,norm_first=True),layers,enable_nested_tensor=False)
               if layers>0 else nn.Identity())
        s.att=nn.Linear(d,1); s.out=nn.Sequential(nn.Dropout(p),nn.Linear(2*d,1))
    def forward(s,x):
        h=s.enc(s.drop(F.gelu(s.proj(s.norm(x))))+s.pos)
        a=torch.softmax(s.att(h).squeeze(-1),1).unsqueeze(-1)     # attention over time
        return s.out(torch.cat([(a*h).sum(1),h.mean(1)],-1)).squeeze(-1)

def train_head(Xtr,ytr,eval_sets,cfg,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    mu,sd=float(ytr.mean()),float(ytr.std()); yt=torch.tensor((ytr-mu)/sd,dtype=torch.float32,device=DEVICE)
    net=Head(Xtr.shape[-1],cfg['d'],cfg['p'],cfg['layers']).to(DEVICE)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg['lr'],weight_decay=cfg['wd'])
    n=len(Xtr); spe=int(np.ceil(n/cfg['bs']))
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=cfg['lr'],total_steps=cfg['epochs']*spe,pct_start=0.1)
    for _ in range(cfg['epochs']):
        net.train(); perm=torch.randperm(n,device=DEVICE)
        for i in range(0,n,cfg['bs']):
            b=perm[i:i+cfg['bs']]; x=Xtr[b].float()
            x=x*(torch.rand(x.shape[0],x.shape[1],1,device=DEVICE)>cfg['fdrop']).float()   # random frame dropout
            loss=F.mse_loss(net(x),yt[b]); opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(),1.0); opt.step(); sch.step()
    net.eval(); outs=[]
    with torch.no_grad():
        for Xe in eval_sets:
            outs.append(np.concatenate([net(Xe[i:i+128].float()).cpu().numpy() for i in range(0,len(Xe),128)])*sd+mu)
    return outs

def cv_nn(cfg,seeds=(0,)):
    oof=np.zeros(len(yg)); tp=np.zeros(len(FR_te))
    for s in seeds:
        for tr,va in folds:
            tr_,va_=torch.as_tensor(tr,device=DEVICE),torch.as_tensor(va,device=DEVICE)
            pv,pt=train_head(FR_g[tr_],yg[tr],[FR_g[va_],FR_te],cfg,s)
            oof[va]+=pv/len(seeds); tp+=pt/(len(folds)*len(seeds))
    return oof,tp

In [ ]:
# Compare two head variants (1 seed, same folds); pick the one that helps the blend most
CFG={'attn':dict(d=128,p=0.3,layers=0,lr=1e-3,wd=0.05,epochs=40,bs=32,fdrop=0.15),
     'tf'  :dict(d=128,p=0.3,layers=1,lr=5e-4,wd=0.05,epochs=40,bs=32,fdrop=0.15)}
cur=rmse(yg,BASE[0]); print('current blend CV:',round(cur,4))
NN_RES={}
for name,cfg in CFG.items():
    o,t=cv_nn(cfg); OOF['nn_'+name]=o; TP['nn_'+name]=t
    bo,_=blend_oof(base_names+['nn_'+name]); NN_RES[name]=(rmse(yg,clip(o)),rmse(yg,bo))
    print(f'{name}: head alone CV {NN_RES[name][0]:.4f} | blend with head {NN_RES[name][1]:.4f} | corr with base blend {np.corrcoef(o,BASE[0])[0,1]:.3f}')
BEST_NN=min(NN_RES,key=lambda k:NN_RES[k][1]); print('selected variant:',BEST_NN)

In [ ]:
# Final head: 3 seeds, fold-averaged test predictions + full-data refit; adopt only if blend CV improves by >= 0.005
cfg=CFG[BEST_NN]; seeds=(0,1,2)
o,tp=cv_nn(cfg,seeds)
ftr=np.zeros(len(yg)); fte=np.zeros(len(FR_te))
for s in seeds:
    a,b=train_head(FR_g,yg,[FR_g,FR_te],cfg,s); ftr+=a/len(seeds); fte+=b/len(seeds)
OOF['nn'],TP['nn'],FULL_TR['nn'],FULL_TE['nn']=o,tp,ftr,fte
WITH=blend_full(base_names+['nn'])
gain=rmse(yg,BASE[0])-rmse(yg,WITH[0])
print(f'head alone CV {rmse(yg,clip(o)):.4f}')
print(f'blend CV without head {rmse(yg,BASE[0]):.4f} | with head {rmse(yg,WITH[0]):.4f} | gain {gain:.4f}')
print('weights with head:',WITH[3])
USE_NN = gain>=0.005
oof_bl,tr_bl,te_bl = (WITH[0],WITH[1],WITH[2]) if USE_NN else (BASE[0],BASE[1],BASE[2])
print('USE_NN =',USE_NN)

## 9. Evaluation 


The final system is evaluated using the speaker-grouped validation strategy established earlier.

### Final Results

| Metric | Result |
|---|---:|
| Mean baseline RMSE | 1.0137 |
| Speaker-grouped CV RMSE | **0.5391** |
| Pearson correlation | **0.847** |
| All-training CV RMSE | **0.5260** |
| Training RMSE | **0.0961** |
| All-training RMSE | **0.0938** |

The final grouped-CV RMSE of **0.5391** represents a substantial improvement over the mean-prediction baseline of **1.0137**.

The Pearson correlation of **0.847** also indicates a strong positive relationship between predicted and actual grammar scores.

### Training RMSE

The competition requires the training RMSE to be reported in the final notebook.

After refitting the final model, the training RMSE is:

> **Training RMSE = 0.0961**

This value is expected to be substantially lower than cross-validation error because the model is evaluated on the same observations used for fitting. Therefore, the grouped-CV RMSE of **0.5391** is considered the more meaningful estimate of generalization performance.

The gap between training and grouped-CV performance also indicates that the final model has substantial capacity relative to the available dataset and should be interpreted with appropriate caution.

In [ ]:
# Metrics on the grammar-model rows, and on ALL training rows (gated clips predicted 0)
full_oof=np.where(GATED[IS_TR],0.0,np.nan); full_trn=full_oof.copy(); y_all=ALL.y.values[IS_TR]
pos_in_tr={ix:i for i,ix in enumerate(np.where(IS_TR)[0])}
for j,ix in enumerate(G_IDX): full_oof[pos_in_tr[ix]]=oof_bl[j]; full_trn[pos_in_tr[ix]]=tr_bl[j]
full_oof=np.nan_to_num(full_oof,nan=yg.mean()); full_trn=np.nan_to_num(full_trn,nan=yg.mean())
print('='*58)
print(f'Baseline (mean) RMSE ................ {rmse(yg,np.full_like(yg,yg.mean())):.4f}')
print(f'Speaker-grouped CV RMSE (grammar rows) {rmse(yg,oof_bl):.4f} | Pearson {pearsonr(yg,oof_bl)[0]:.3f}')
print(f'Speaker-grouped CV RMSE (ALL train) .. {rmse(y_all,full_oof):.4f}')
print(f'TRAINING RMSE (grammar rows, refit) .. {rmse(yg,tr_bl):.4f}')
print(f'TRAINING RMSE (ALL train rows) ....... {rmse(y_all,full_trn):.4f}')
print('='*58)
fig,ax=plt.subplots(1,3,figsize=(16,4))
ax[0].scatter(yg+np.random.normal(0,.04,len(yg)),oof_bl,alpha=.4); ax[0].plot([LO,5],[LO,5],'r--'); ax[0].set(xlabel='true',ylabel='OOF prediction',title='Out-of-fold predictions')
ax[1].hist(yg-oof_bl,bins=30); ax[1].set_title('Residuals')
pd.DataFrame({'t':np.round(yg).astype(int),'e':np.abs(yg-oof_bl)}).groupby('t').e.mean().plot.bar(ax=ax[2],title='Mean |error| by true grade'); plt.tight_layout(); plt.show()

In [ ]:
# Interpretability: which interpretable features drive the tree model? (tabular blocks only)
tabcols=sum([COLS[b] for b in TAB],[]); m=lgbm().fit(Xg[tabcols],yg)
imp=pd.Series(m.booster_.feature_importance('gain'),index=tabcols).sort_values(ascending=False)[:15][::-1]
imp.plot.barh(figsize=(6,5),title='LightGBM gain importance (interpretable features)'); plt.show()
print(pd.DataFrame({'pearson_with_label':Xg[tabcols].corrwith(pd.Series(yg))}).sort_values('pearson_with_label').iloc[[0,1,2,3,-4,-3,-2,-1]])

## 10. Test predictions & submission


In [ ]:
pred=np.zeros(len(ALL)); pred[np.where(IS_TE)[0]]=te_bl
pred[GATED&IS_TE]=0.0                                    # corrupted-audio gate
sub=pd.DataFrame({sub_ex.columns[0]:test[FILE_COL].values,sub_ex.columns[1]:pred[IS_TE]})
sub.to_csv('submission.csv',index=False)
print(sub.shape,'| NaN:',sub.isna().sum().sum(),'| range:',round(float(sub.iloc[:,1].min()),3),round(float(sub.iloc[:,1].max()),3))
sub.head()

## 11. Results summary 

### Public Leaderboard Result

The previous submitted configuration achieved a public leaderboard RMSE of
**0.3784**.

The final attention-refined configuration achieved **0.3728**, representing
an improvement of **0.0056 RMSE**.

This provides independent evidence that the grouped-CV improvement was not
purely caused by the validation procedure.

In [ ]:
print('Mean-predictor RMSE ..................', round(rmse(yg,np.full_like(yg,yg.mean())),4))
print('Random vs grouped CV (Ridge, all blocks)', round(RG[0],4),'vs',round(RG[1],4))
print('Single-model CV ......................', {k:round(rmse(yg,clip(OOF[k])),4) for k in base_names})
print('Base blend CV (no head) ..............', round(rmse(yg,BASE[0]),4),'| weights',BASE[3])
if 'WITH' in globals():   # only exists if the attention-head cells (Section 8b) were run
    print('Blend CV with attention head .........', round(rmse(yg,WITH[0]),4),'| head alone',round(rmse(yg,clip(OOF['nn'])),4),'| used:',USE_NN)
print('FINAL speaker-grouped CV RMSE ........', round(rmse(yg,oof_bl),4),'| all-train', round(rmse(y_all,full_oof),4),'| Pearson',round(pearsonr(yg,oof_bl)[0],3))
print('FINAL TRAINING RMSE ..................', round(rmse(yg,tr_bl),4),'| all-train', round(rmse(y_all,full_trn),4))

## 12. Limitations and Potential Sources of Optimism

1. **Model selection on validation folds:** Feature blocks, model choices and
   blend weights were selected using the same grouped-CV framework, so the
   reported CV estimate may be somewhat optimistic.

2. **Inferred speaker groups:** Speaker identity is not explicitly provided.
   Groups are inferred from low-level speech representations.

3. **ASR dependency:** Transcription errors can alter downstream grammar
   features and may smooth over genuine grammatical mistakes.

4. **Accent sensitivity:** Whisper transcription quality may vary across
   speakers and accents.

5. **Train/CV gap:** The low training RMSE relative to grouped CV indicates
   substantial overfitting capacity.

## 14. Final Model Summary

The final Grammar Scoring Engine follows the pipeline:

**Audio → Whisper-medium → Linguistic + acoustic + semantic representations → Regression ensemble → Temporal attention refinement → 0–5 grammar score**

### Final Configuration

| Component | Final choice |
|---|---|
| ASR | Whisper-medium |
| Validation | 5-fold speaker-grouped CV |
| Linguistic features | Fluency, lexical, grammar and error statistics |
| Speech features | wav2vec2 + Whisper representations |
| Semantic features | Sentence embeddings |
| Base models | Ridge + LightGBM + SVR |
| Ensemble | Non-negative OOF blending |
| Temporal model | Attention-pooling head |
| Output | Clipped to 0–5 |

### Final Performance

**Grouped CV RMSE:** 0.5391  
**Pearson correlation:** 0.847  
**Training RMSE:** 0.0961  
**Public leaderboard RMSE:** 0.3728

The final system combines explicit grammar signals with pretrained speech and language representations, while speaker-grouped validation provides a conservative estimate of performance on unseen speakers.

## References

1. Radford et al. — *Robust Speech Recognition via Large-Scale Weak Supervision* (Whisper).
2. Baevski et al. — *wav2vec 2.0: A Framework for Self-Supervised Learning of Speech Representations*.
3. Radford et al. — *Language Models are Unsupervised Multitask Learners* (GPT-2).
4. Warstadt et al. — *Neural Network Acceptability Judgments* / CoLA.
5. Sentence-Transformers — MPNet sentence representations.
6. LightGBM documentation and implementation.
7. scikit-learn documentation.
8. PyTorch documentation.
9. LanguageTool.